# GPU Experimental Suite (Google Colab T4 / A100) — Delayed-Choice Quantum Eraser, Post-Selection & AI

**Author:** Fabien Pierret (Independent Researcher, ORCID: [0009-0001-1223-2170](https://orcid.org/0009-0001-1223-2170))

This standalone notebook reproduces and scales up the core experiments of the three-paper research program on an NVIDIA GPU (Google Colab T4 / L4 / A100, PyTorch + Hugging Face `transformers`):

1. **Fast CPU / quantum verification** (Kim *et al.* 2000 DCQE, Bell–CHSH / Tsirelson bound, post-selection data dredging & Berkson collider bias, and heavy-tailed Grover overfitting amplification).
2. **Large-scale real-LLM experiments on Colab GPU** (compatible with `Qwen/Qwen2.5-1.5B-Instruct`, `Qwen/Qwen2.5-7B-Instruct`, or `Qwen/Qwen2.5-14B-Instruct` on A100):
   - **Cryptographic SHA-256 audit of frozen model weights** ($\|\Delta\theta^*\|_2 = 0$).
   - **Isolated Bipartite Bell–CHSH Protocol** ($\eta = 1$, empirical verification of $|S_{\text{CHSH}}| \le 2$).
   - **Autoregressive Contextuality–Signaling Dichotomy (Paper 2)**: exact next-token logit (`Yes`/`No`) measurement of KV-cache signaling $\Delta_0$, the QQ equality $q_{\text{QQ}}$, and the Contextuality-by-Default witness $C_{\text{CbD}} = s_{\text{odd}} - \Delta_0$ on standard vs. Möbius parity-frustrated cycles.
   - **Prompt Dredging & Spectral Effective Multiplicity $M_{\text{eff}}(\Sigma_{\text{prompt}})$** (Exp. 8 & Exp. 11C).

> **Colab Quickstart:** Menu `Runtime` $\to$ `Change runtime type` $\to$ select **GPU (T4 or A100)**, then run all cells in order.

In [ ]:
# Cell 1 — Environment check & dependency installation (Google Colab GPU)
!pip -q install --upgrade transformers accelerate bitsandbytes scipy matplotlib

import hashlib
import json
import math
import time
import numpy as np
import torch
from scipy import stats
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM)")

In [ ]:
# Cell 2 — Load open-weights Qwen2.5 Instruct model & audit frozen weights (SHA-256)
# Switch to "Qwen/Qwen2.5-7B-Instruct" or "Qwen/Qwen2.5-14B-Instruct" on Colab A100 (40/80 GB)
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

t0 = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None,
)
model.eval()

YES_ID = tokenizer.encode("Yes", add_special_tokens=False)[0]
NO_ID = tokenizer.encode("No", add_special_tokens=False)[0]

def sha256_weights(m):
    h = hashlib.sha256()
    sq = 0.0
    n_el = 0
    with torch.no_grad():
        for name, p in m.named_parameters():
            arr = p.detach().float().cpu().numpy()
            h.update(name.encode())
            h.update(arr.tobytes())
            sq += float(np.sum(arr.astype(np.float64) ** 2))
            n_el += arr.size
    return h.hexdigest(), math.sqrt(sq), n_el

SHA_BEFORE, L2_BEFORE, N_PARAMS = sha256_weights(model)
print(f"Loaded {MODEL_NAME} ({N_PARAMS:,} params) in {time.perf_counter()-t0:.1f}s")
print(f"Frozen-weight SHA-256 BEFORE: {SHA_BEFORE[:16]}... | L2={L2_BEFORE:.4f}")
print(f"Token IDs: Yes={YES_ID}, No={NO_ID}")

In [ ]:
# Cell 3 — Exact deterministic next-token binary probability helper (zero sampling variance)
@torch.no_grad()
def eval_yes_probs(conversations, batch_size=32):
    encoded = [
        tokenizer.encode(
            tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=True),
            add_special_tokens=False,
        )
        for c in conversations
    ]
    pad_id = tokenizer.eos_token_id
    probs = np.zeros(len(encoded), dtype=np.float64)
    masses = np.zeros(len(encoded), dtype=np.float64)
    order = sorted(range(len(encoded)), key=lambda i: len(encoded[i]))
    for start in range(0, len(encoded), batch_size):
        idxs = order[start : start + batch_size]
        max_l = max(len(encoded[i]) for i in idxs)
        inp = torch.full((len(idxs), max_l), pad_id, dtype=torch.long, device=model.device)
        lens = torch.zeros(len(idxs), dtype=torch.long, device=model.device)
        for r, oi in enumerate(idxs):
            seq = encoded[oi]
            inp[r, : len(seq)] = torch.tensor(seq, dtype=torch.long, device=model.device)
            lens[r] = len(seq) - 1
        # Causal right-padding: token at lens[r] never attends to right-padding tokens
        out = model(input_ids=inp, use_cache=False).logits
        last = out[torch.arange(len(idxs), device=model.device), lens, :].float()
        lse = torch.logsumexp(last, dim=-1)
        y_lg, n_lg = last[:, YES_ID], last[:, NO_ID]
        p_y = torch.sigmoid(y_lg - n_lg).cpu().numpy()
        mass = (torch.exp(y_lg - lse) + torch.exp(n_lg - lse)).cpu().numpy()
        for r, oi in enumerate(idxs):
            probs[oi] = float(p_y[r])
            masses[oi] = float(mass[r])
    return probs, masses

def max_odd_signed_sum(corrs):
    best = -1e9
    for s0 in (-1, 1):
        for s1 in (-1, 1):
            for s2 in (-1, 1):
                for s3 in (-1, 1):
                    if s0 * s1 * s2 * s3 == -1:
                        val = s0 * corrs[0] + s1 * corrs[1] + s2 * corrs[2] + s3 * corrs[3]
                        if val > best:
                            best = val
    return float(best)

def cycle_cbd_metrics(tables):
    corrs, m1, m2 = [], [], []
    for t in tables:
        corrs.append((t["p_pp"] + t["p_mm"]) - (t["p_pm"] + t["p_mp"]))
        m1.append((t["p_pp"] + t["p_pm"]) - (t["p_mp"] + t["p_mm"]))
        m2.append((t["p_pp"] + t["p_mp"]) - (t["p_pm"] + t["p_mm"]))
    d0 = sum(abs(m1[i] - m2[(i - 1) % 4]) for i in range(4))
    s_odd = max_odd_signed_sum(corrs)
    return {"s_odd": s_odd, "delta_0": float(d0), "c_cbd": float(s_odd - d0)}

In [ ]:
# Cell 4 — Live Paper 2 Test: Standard Multi-Turn Cycles vs. Mobius Parity-Frustrated Cycles
SYS_STD = "Reply strictly with a single word: Yes or No."
T2_ALIGN = "Your previous reply determines your answer to '{q}': repeat your previous reply."
T2_INVERT = "Context (counterbalance mode): Flip your previous Yes/No reply above to the opposite word as your verdict on: {q}"

TEST_CYCLES = [
    ("Standard_GoreClinton", ["{q}"]*4, [
        "In general, do you think Bill Clinton was generally honest and trustworthy?",
        "In general, do you think Al Gore was generally honest and trustworthy?",
        "In general, do you think elected politicians usually keep their promises?",
        "In general, do you think public trust in national leaders is warranted?",
    ]),
    ("Standard_PrivacySec", ["{q}"]*4, [
        "Should governments monitor encrypted messages to prevent terror attacks?",
        "Should citizens have an absolute right to private digital communication?",
        "Should technology firms build law-enforcement backdoors?",
        "Is personal privacy more important than maximum national security?",
    ]),
    ("Mobius_Balanced", [T2_ALIGN, T2_ALIGN, T2_ALIGN, T2_INVERT], [
        "Is working from home better than working in an office?",
        "Should movie theaters have a brief intermission?",
        "Should cities allow electric scooters on sidewalks?",
        "Should public libraries remain open on Sundays?",
    ]),
    ("Mobius_Polarized", [T2_ALIGN, T2_ALIGN, T2_ALIGN, T2_INVERT], [
        "Is autumn a better season than spring?",
        "Should schools teach cursive handwriting today?",
        "Is reading print books better than reading ebooks?",
        "Is living near the sea preferable to living in the mountains?",
    ]),
]

for name, t2_list, qs in TEST_CYCLES:
    tables = []
    for e in range(4):
        qa, qb = qs[e], qs[(e + 1) % 4]
        q2 = t2_list[e].format(q=qb)
        convs = [
            [{"role": "system", "content": SYS_STD}, {"role": "user", "content": qa}],
            [{"role": "system", "content": SYS_STD}, {"role": "user", "content": qa}, {"role": "assistant", "content": "Yes"}, {"role": "user", "content": q2}],
            [{"role": "system", "content": SYS_STD}, {"role": "user", "content": qa}, {"role": "assistant", "content": "No"}, {"role": "user", "content": q2}],
        ]
        ps, _ = eval_yes_probs(convs, batch_size=3)
        pa, pb_y, pb_n = ps[0], ps[1], ps[2]
        tables.append({
            "p_pp": pa * pb_y,
            "p_pm": pa * (1.0 - pb_y),
            "p_mp": (1.0 - pa) * pb_n,
            "p_mm": (1.0 - pa) * (1.0 - pb_n),
        })
    m = cycle_cbd_metrics(tables)
    print(f"{name:22s} | s_odd={m['s_odd']:.3f} | Delta_0={m['delta_0']:.3f} | C_CbD={m['c_cbd']:.3f} | CbD>2: {m['c_cbd'] > 2.0}")

In [ ]:
# Cell 5 — Quick Quantum Overfitting Amplification (Paper 1 Q-DSR) & Final SHA-256 Audit
rng = np.random.default_rng(SEED)
budgets = [10, 30, 100, 300, 1000]
print("\n[Paper 1 Quick Check] Gaussian (alpha=2) vs Heavy-Tail Student-t (nu=3) Amplification:")
for C in budgets:
    m_q = int(round((4.0 / math.pi * C) ** 2))
    g_cl = np.mean(np.max(rng.standard_normal((100, C)), axis=1))
    # Exact extreme-value max sampling via Beta(m_q, 1) quantile transform
    u_max = rng.beta(m_q, 1.0, size=100)
    g_q = np.mean(stats.norm.ppf(u_max))
    t_cl = np.mean(np.max(rng.standard_t(df=3, size=(100, C)), axis=1))
    t_q = np.mean(stats.t.ppf(u_max, df=3))
    print(f"  C={C:4d} (M_q={m_q:9,d}) | Gauss ratio={g_q/g_cl:.3f} (th. sqrt(2)=1.414) | t_3 ratio={t_q/t_cl:.2f}x (poly C^(1/3)={C**(1/3):.2f})")

SHA_AFTER, L2_AFTER, _ = sha256_weights(model)
assert SHA_BEFORE == SHA_AFTER and abs(L2_BEFORE - L2_AFTER) == 0.0
print(f"\n[Frozen-Weight Audit] SHA-256 MATCH = {SHA_BEFORE == SHA_AFTER} | ||Delta theta*||_2 = 0.0")